# 02 - Two Solitons

This notebook computes two-eigenvalue reflectionless KdV potentials with the stable 2-fold Crum step, checks positivity, and compares against a repeated 1-fold Darboux chain.

In [ ]:
using Pkg
Pkg.activate("..")

using Revise
includet("../src/KdVCrumProject.jl")
using .KdVCrumProject

using Plots, LaTeXStrings

For a regular symmetric two-soliton example, use `beta_n=(-1)^(N-n)`. The spectrum must be strictly increasing.

In [ ]:
x = collect(range(-20.0, 20.0; length=3001))
kappa = [0.8, 1.7]
beta = alternating_norming_constants(length(kappa))

result = crum_transform(x, kappa, beta; return_history=true)
q = result.q

E1, E2 = conserved_errors(x, q, kappa)

@show beta
@show all(isfinite, q)
@show minimum(q)
@show maximum(q)
@show E1 E2;

In [ ]:
plot_potential(x, q; title="Stable two-soliton Crum transform", label=L"q_2(x)")

The step history records which fold was applied and the sampled range of the intermediate potential.

In [ ]:
for step in result.history
    println("fold=", step.fold,
            " kappa=", step.kappa,
            " min_q=", step.min_q,
            " max_q=", step.max_q)
end

The naive Darboux chain is analytically equivalent in exact arithmetic, but it is the reference path that can create artificial singularities for larger or less favorable spectra.

In [ ]:
q_naive_smallest = naive_darboux_chain(x, kappa, beta; order=:smallest_first)
q_naive_largest = naive_darboux_chain(x, kappa, beta; order=:largest_first)

@show all(isfinite, q_naive_smallest)
@show all(isfinite, q_naive_largest)
@show relative_l2_error(q_naive_smallest, q)
@show relative_l2_error(q_naive_largest, q)

plt = plot_potential(x, q; label="stable 2-fold", title="Stable vs naive Darboux chains")
plot!(plt, x, q_naive_smallest; label="naive smallest-first", linestyle=:dash)
plot!(plt, x, q_naive_largest; label="naive largest-first", linestyle=:dot)
plt

A shifted/asymmetric two-soliton example is obtained by passing one shift per eigenvalue. Internally this preserves the signs and uses `beta_n * exp(2*kappa_n*shift_n)`.

In [ ]:
shifts = [-2.5, 1.5]
q_shifted = crum_transform(x, kappa, beta; shifts=shifts)
beta_shifted = shifted_norming_constants(kappa, beta, shifts)

@show shifts
@show beta_shifted
@show minimum(q_shifted)
@show maximum(q_shifted)
@show conserved_errors(x, q_shifted, kappa)

plot_potential(x, q_shifted; title="Shifted two-soliton example", label=L"q_2(x)")

Time evolution keeps `kappa` fixed and evolves the norming constants by `beta_n(t)=beta_n(0)*exp(8*kappa_n^3*t)`. The larger soliton therefore travels faster.

In [ ]:
times = range(-2.0, 2.0; length=81)

anim = @animate for t in times
    beta_t = evolve_beta(kappa, beta, t)
    q_t = crum_transform(x, kappa, beta_t)
    plot_potential(
        x,
        q_t;
        title="Two-soliton evolution, t=$(round(t; digits=2))",
        label=L"q(x,t)",
        ylims=(0, 1.1 * maximum(q)),
        legend=:topright,
    )
end

gif(anim, "two_soliton_evolution.gif"; fps=20)

For a single time slice, use the same evolution formula without making an animation.

In [ ]:
t = 1.0
beta_t = evolve_beta(kappa, beta, t)
q_t = crum_transform(x, kappa, beta_t)

@show beta_t
plot_potential(x, q_t; title="Two-soliton at t=$t", label=L"q(x,t)")